<!-- nav -->
[← 4 · A family tree: self-relations, cycles and reachability](04_A_Family_Tree.ipynb) · [Home](../../README.md) · [6 · Saving and loading: documents →](06_Saving_And_Loading.ipynb)

# 5 · A price list: cardinality and validation

**The problem.** An online shop sells each product in several currencies. A product has *at most one* price per
currency, and a price object belongs to *exactly one* product and currency. Stock items belong to at most one
warehouse. Data arrives from spreadsheets and partner feeds, and it's often wrong in small ways: a price given as the
string `"12.99"`, two euro prices for the same product.

Two questions follow:

1. How do you *say* "at most one price per currency" in the schema?
2. When and how is data *checked* against it?

The answers: `unique(...)` clauses on relations, given with `.uniques(...)`, and a validator you call when you choose
to.

In [1]:
from mbse.Schemas.Framework import Builders, Schemas, Validators

<a id="T05.1"></a>
## 05.1 · A dictionary is a relation with a key

In Python you might write `product.prices: dict[str, Money]`. In this framework a map `product -> [currency] ->
price` is a relation with links `product` and `price` and a property `currency`. It's the same shape you've used
since case study 2. The key is simply an entry property.

In [2]:
def install(txn):
    """The price list's schemas: products, prices in money, and the relation between them."""
    S = txn.Schemas  # the store's own schemas, by name
    return txn.uid('tutorial.prices/1').mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({'Product': S.OfObject, 'Money': S.OfObject, 'Prices': S.OfRelation})
        .spec([
            lambda _: _.Product.name('Product').ref(True).properties({'sku': S.OfStr}).adjacencies({
                'prices': lambda a: a.relation([_.Prices]).me('product')}),
            lambda _: _.Money.name('Money').ref(True).properties({
                'minor_units': S.OfInt,  # cents
            }).adjacencies({'priced': lambda a: a.relation([_.Prices]).me('price')}),
            lambda _: _.Prices.name('Prices').links([
                lambda t: t.name('product'),
                lambda t: t.name('price'),
            ]).properties({
                'currency': S.OfStr,  # ISO 4217, e.g. 'EUR'
            }).uniques([
                ['price'],                # (product, currency) determine the price: one price per currency
                ['product', 'currency'],  # the price determines (product, currency): a price is used once
            ]),
        ])]))])


store = Builders.Store()
store.transaction(install)
Product, Money, Prices = (store.registered(name) for name in ('Product', 'Money', 'Prices'))
[sorted(clause) for clause in Prices.uniques]  # each clause, a set of names

[['price'], ['currency', 'product']]

<a id="T05.2"></a>
## 05.2 · Reading `unique(S)`

`unique(S)` names the things that are *determined*. Everything **not** in `S`, taken together, is a key: two entries
that agree on everything outside `S` must also agree on `S`. A relation's `.uniques(...)` gives its clauses, naming
links and properties by string: `.uniques('price')` is one clause of one name, `.uniques(['product', 'currency'])`
one clause of two, and a list of such lists several clauses, as `Prices` has.

| Clause on `Prices` | The key (everything else) | In words |
|---|---|---|
| `unique(price)` | `product`, `currency` | a product has at most one price per currency |
| `unique(product, currency)` | `price` | a price object is used by at most one (product, currency) |

**Why this notation, and not "one-to-many" labels?** Real relations have more than two ends and carry properties,
which "one-to-many" can't describe; naming what the rest determines covers every case, a clause per constraint.
Ownership, keyed ownership and directories are such combinations of clauses, listed in
[CAP-155: `unique` clauses on relation schemas govern relation cardinality](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-155.md).

`unique` gives only an upper bound. "At least one" is a constraint of another kind, written with mbse-patterns
([CAP-152: relation constraint terms as mandatory, possible, or forbidden entry patterns](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-152.md)). Nothing is mandatory until a constraint says so, as in case study 1.

Warehouses come in an install of their own, `stock`. It names `Product` too: the store finds it by its name, and it
gains the products' side of the new relation. An install adds to what exists; run again, it writes nothing.

In [3]:
def stock(txn):
    """Warehouses and what they stock; it adds to Product, which `install` made, the products' side of it."""
    S = txn.Schemas
    return txn.uid('tutorial.stock/1').mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({'Warehouse': S.OfObject, 'Stocking': S.OfRelation, 'Product': S.OfObject})
        .spec([
            lambda _: _.Warehouse.name('Warehouse').ref(True).properties({'city': S.OfStr}).adjacencies({
                'stock': lambda a: a.relation([_.Stocking]).me('warehouse')}),
            lambda _: _.Stocking.name('Stocking').links([
                lambda t: t.name('warehouse'),
                lambda t: t.name('product'),
            ]).uniques('warehouse'),  # ownership
            lambda _: _.Product.name('Product').adjacencies({
                'stored_in': lambda a: a.relation([_.Stocking]).me('product')}),
        ])]))])


store.transaction(stock)  # finds Product by its name, and adds to it
Warehouse, Stocking = store.registered('Warehouse'), store.registered('Stocking')

for schema in [Product, Money, Prices, Warehouse, Stocking]:
    assert schema.validate() == [], (schema.name, schema.validate())

`Schema.validate()` also checks the clauses themselves. For example, a clause may only name links and properties the
relation has:

In [4]:
def unknown(txn):
    """A relation whose clause names a link it doesn't have."""
    return txn.uid('tutorial.unknown/1').mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({'Unknown': txn.Schemas.OfRelation})
        .spec([lambda _: _.Unknown.name('Unknown').links([
            lambda t: t.name('a'),
            lambda t: t.name('b'),
        ]).uniques('c')])]))])


store.transaction(unknown)
store.registered('Unknown').validate()

["unique(c) names unknown links or properties ['c']"]

<a id="T05.3"></a>
## 05.3 · Good data

A validator is constructed with the *store*, which it uses to look up schemas by the names objects carry. Call it
with a schema and a value. It returns a list of problems,
empty when all is well:

In [5]:
validate = Validators.Validate(store)

with store.transaction():
    widget = store.Product(lambda p: (
        p
        .sku('W-1')
        .prices(lambda a: a.insert(lambda x: x.price(lambda m: m.minor_units(1299)).currency('EUR')))
        .prices(lambda a: a.insert(lambda x: x.price(lambda m: m.minor_units(1500)).currency('JPY')))
    ))
validate(Product, widget)

[]

<a id="T05.4"></a>
## 05.4 · Bad data, all of it at once

A partner feed sends a gadget with two euro prices, one of which was written as a string:

In [6]:
with store.transaction():
    gadget = store.Product(lambda p: (
        p
        .sku('G-1')
        .prices(lambda a: a.insert(lambda x: x.price(lambda m: m.minor_units(900)).currency('EUR')))
        .prices(lambda a: a.insert(lambda x: x.price(lambda m: m.minor_units('950')).currency('EUR')))
    ))
for problem in validate.Reachable(Product, gadget):
    print(problem)

Money#2.minor_units: expected int, got str
unique(price) violated: entries agreeing on ['currency', 'product'] differ on ['price']


Each problem says where it is. `Money#2.minor_units` is the `minor_units` of the object numbered 2 in the order the
validator reached them, and a problem inside an entry is written as a path like `Product#0.prices[1].currency`.
The uniqueness message names the key the entries agree on and what they disagree about.

`validate(schema, value)` checks the value and its entries. `validate.Reachable(schema, root)` checks everything
reachable from the root, as defined in case study 4. That's how the bad `Money` object was found.

### Why doesn't the builder stop this?

Case study 1 showed that builders accept any value. Validation is on request so that you get every problem at once,
not the first; intermediate states stay legal; and you choose when to pay for checks that span many objects
([CAP-160: data validated against its schema on request](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-160.md), applying [PRI-007: three-valued checking on request](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-007)).

The trade-off: data is only known to be valid after you've validated it. Validate at trust boundaries: after an
import, before a save, before handing data to another system.

<a id="T05.5"></a>
## 05.5 · Fixing it

Using case study 3's removal, drop the entry whose price is the string:

In [7]:
from toolkit import remove_entries

with store.transaction():
    remove_entries(gadget, 'prices', where=lambda row: isinstance(row['price'].minor_units.get(), str))
validate.Reachable(Product, gadget)

[]

<a id="T05.6"></a>
## 05.6 · Links must be filled by the right kind of object

Links have no schemas in the relation, but the *objects* declare which links they can fill, each through an adjacency:
named, when the object reads its entries through it, or anonymous, given in a list
(`.adjacencies([lambda a: a.relation([_.Relation]).me(link)])`), when it needs no member for them. The validator checks
each linked object's schema against that. Here a `Warehouse` has been linked where a price belongs. The builder allowed
it, because an existing object can always be linked, and the validator catches it:

In [8]:
with store.transaction():
    berlin = store.Warehouse(lambda w: w.city('Berlin'))
    store.Product(widget, lambda p: p.prices(lambda a: a.insert(lambda x: x.price(berlin).currency('GBP'))))
for problem in validate(Product, widget):
    print(problem)
with store.transaction():
    widget = remove_entries(widget, 'prices', where=lambda row: row['price'] is berlin)  # returns the updated object

Product#0.prices[2].price: a 'Warehouse' cannot fill link 'price'; its schema declares no adjacency, named or anonymous, to this relation via 'price'


<a id="T05.7"></a>
## 05.7 · Ownership

`Stocking` says a product is in at most one warehouse. Stocking the widget in two breaks it:

In [9]:
with store.transaction():
    paris = store.Warehouse(lambda w: w.city('Paris'))
    store.Warehouse(berlin, lambda w: w.stock(lambda a: a.insert(lambda x: x.product(widget))))
    store.Warehouse(paris, lambda w: w.stock(lambda a: a.insert(lambda x: x.product(widget))))
validate.Reachable(Product, widget)

["unique(warehouse) violated: entries agreeing on ['product'] differ on ['warehouse']"]

Uniqueness is checked over the entries the validator sees, which with `Reachable` is the whole connected component.
(Entries in a separate component that happen to share property values are not compared. Whether a relation-wide
check is needed is an open question in [CAP-160: data validated against its schema on request](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-160.md#open-questions).)

<a id="T05.8"></a>
## 05.8 · Validating a single value

The validator works for any schema, not just objects. This is handy in form handlers:

In [10]:
print(validate(store.Schemas.OfInt, 3), validate(store.Schemas.OfInt, True))

[] ['expected int, got bool']


`True` isn't an `int` here, even though Python says `isinstance(True, int)`. Scalars are exact and never
coerced
into each other, in validation, in serialization and in equality. `1`, `1.0` and `True` are three different values.
That's what lets the same data mean the same thing in a language where they really are different host types.

## What you learned

- Maps are relations with a key property. `unique(S)` says the rest of the entry determines `S`; `.uniques(...)`
  gives a relation's clauses, naming links and properties by string.
- Ownership, keyed ownership and directories are combinations of `unique` clauses.
- `Validators.Validate(store)` checks data only when called, and returns every problem with a path.
  `.Reachable` checks a whole component.
- Linked objects must have a schema that declares the link they fill, by a named or an anonymous adjacency.
- Scalars are exact: `True` is not an `int`, and `1.0` is not `1`.

**Next:** the data is right. Case study 6 saves it and loads it back, which is where the insistence on relations,
schema names and exact scalars pays off.

<!-- nav -->
[← 4 · A family tree: self-relations, cycles and reachability](04_A_Family_Tree.ipynb) · [Home](../../README.md) · [6 · Saving and loading: documents →](06_Saving_And_Loading.ipynb)